In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
from scipy.stats import gaussian_kde
from itertools import chain
from matplotlib.colors import ListedColormap
from scipy.stats import gaussian_kde

import rasterio as rio
from rasterio.plot import show
from rasterio.mask import mask

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib inline

from RGB_extract_aux import *

In [5]:
# select all sites that should be included in the taining dataset (RGB_rcgc_chm_dataset.csv file)
sites = ['CS3A']

# all sites
#sites = ['CG1-8A', 'CG1-8B', 'CS3A', 'CS3B', 'CS-46A', 'CS-46B', 'CS-59B', 'CS-96B', 'CS-103B', 'CS117B', 
#         'F3-8A', 'F3-8B', 'F3-8C', 'F3-20A', 'F3-20B', 'F3-20C', 'F3-19B', 'F3-19C', 
#         'ZF20-11A', 'ZF46-15A', 'ZF46-37A']

In [7]:
RGB_all = {}
LatLon_all = {}
RcGc_all = {}
CHM_all = {}

# include all classes (names and corresponding values) that should be included in the training dataset (RGB_rcgc_chm_dataset.csv file)
classnames = ['lichen', 'rock', 'broadleaf', 'needleleaf', 'deadwood', 'graminoids', 
              'moss', 'soil', 'low_green', 'dry_branches']
classvals = [1, 4, 5, 6, 7, 8, 9, 10, 12, 13]

for site in sites:
    print(site)
    # LABELLED POLYGONS
    # ----------------------------------------------------------------------------------------------------------
    # load labelled polygons as geodataframe with geopandas and filter them based on their label, eg. lichen = 1, moss = 9
    # using auxilary function label_filter() in RGB_extract_functions (same dirctory as notebook).
    # ----------------------------------------------------------------------------------------------------------
    #wd = "C:/Users/MABEB16/science/postdoc/results/projects/lichen_mapping/classification/qgis_%s_hp/" %site
    gdf = gpd.read_file(r'data/%s_hp_labelledpoints.gpkg' %site, layer='%s_hp_labelledpoints' %site)
    filtered_gdf = label_filter(gdf=gdf, classnames=classnames, classvals=classvals)
    
    # MISSION RASTER and LAT-LON arrays
    # ----------------------------------------------------------------------------------------------------------
    # load ortho raster file for the site defined in site parameter above. Loaded using rasterio
    # ----------------------------------------------------------------------------------------------------------
    # RGB raster
    ortho = "data/%s_hp_transparent_mosaic_group1.tif" % site
    ortho_raster = rio.open(ortho)
    ortho_meta = (ortho_raster.crs.data, ortho_raster.width, ortho_raster.height)

    # lons and lats arrays
    band1 = ortho_raster.read(1)
    height = band1.shape[0]
    width = band1.shape[1]
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rio.transform.xy(ortho_raster.transform, rows, cols)
    lons= np.array(xs)
    lats = np.array(ys)
    
    # EXTRACT RGB, LAT, LON, Rc, Gc
    # ----------------------------------------------------------------------------------------------------------
    # use filtered polygons in filtered_gdf to extract RGB, lat, lon, Rc and Gc values of labelled pixel per class in the raster
    # using auxilary function extractDataPerLabel() in RGB_extract_functions (same dirctory as notebook).
    # ----------------------------------------------------------------------------------------------------------
    RGB, LatLon, RcGc  = extractDataPerLabel(raster=ortho_raster, 
                                             filtergdf=filtered_gdf, 
                                             lon_array=lons, 
                                             lat_array=lats, 
                                             classnames=classnames)
    RGB_all[site] = RGB
    LatLon_all[site] = LatLon
    RcGc_all[site] = RcGc
    
    # CHM polygons
    # ----------------------------------------------------------------------------------------------------------
    # load chm polygone shapefile for the site defined in site parameter above. Loaded using geopandas dataframe
    # ----------------------------------------------------------------------------------------------------------
    chm = "%sstratified_sampling/%s_hp_chm_stratified.shp" % (wd, site)
    chm_gdf = gpd.GeoDataFrame.from_file(chm)
    
    # EXTRACT chm strata values
    # ----------------------------------------------------------------------------------------------------------
    # use filtered polygons in lichensub to extract CHM strata values of labelled pixel per class
    # using the function extractCanopyHeightPerLabel stored in the auxilary file RGB_extract_functions in the same dirctory as notebook
    # ----------------------------------------------------------------------------------------------------------
    chm_data = extractCanopyHeightPerLabel(raster=ortho_raster, 
                                           chm_gdf=chm_gdf, 
                                           filtergdf=filtered_gdf, 
                                           classnames=classnames)
    
    CHM_all[site]=chm_data

CS3A


C:\Users\MABEB16\Anaconda3\envs\classification\Lib\site-packages\rasterio\mask.py:88: UserWarning: shapes are outside bounds of raster. Are they in different coordinate reference systems?
  warnings.warn('shapes are outside bounds of raster. '


KeyboardInterrupt: 

In [8]:
# save extracted RGB, chm and calculated rc and gc values together with class and site information in .csv file. This is file is the base
# for further preprocessing steps and the training data for the classifier.
all_list = []
for site in sites:
    for veg_class in classnames:
        for i, val in enumerate(RGB_all[site][veg_class]['red']):
            all_tmp = [site, veg_class, 
                       val/255, RGB_all[site][veg_class]['green'][i]/255, RGB_all[site][veg_class]['blue'][i]/255, 
                       RcGc_all[site][veg_class]['rc'][i], RcGc_all[site][veg_class]['gc'][i], CHM_all[site][veg_class][i]]
            all_list.append(all_tmp)
            
df = pd.DataFrame.from_dict(all_list)
df.columns =['site', 'veg_class', 'R', 'G', 'B', 'rc', 'gc', 'chm']



filename = 'data/RGB_rcgc_chm_dataset.csv'
df.to_csv(filename, sep=',', index=False)

KeyError: 'CS3A'